# Training CNN-LSTM RUL (FD004)

Runner del training: richiama la funzione `train(...)` definita in `src/train.py`, cosi non duplica la logica del modello.

Usa i dati locali in `CMAPPS-data/` e salva gli artefatti in `outputs/`. Gira nel kernel corrente.

In [ ]:
import importlib
import sys
from pathlib import Path

# Su Compute Instance il notebook gira da notebooks/, con src/ e CMAPPS-data/ accanto.
REPO_ROOT = Path.cwd().resolve().parent
SRC_DIR = REPO_ROOT / 'src'
DATA_FILE = REPO_ROOT / 'CMAPPS-data' / 'train_FD004.txt'
OUTPUT_DIR = REPO_ROOT / 'outputs'

print('Repo root:', REPO_ROOT)
print('Dati:', DATA_FILE)

# Rendi importabile il codice di training
sys.path.insert(0, str(SRC_DIR))
importlib.reload(train)
from train import train



In [ ]:
# Lancia il training (gli iperparametri hanno gli stessi default del job AML).
# use_mlflow=False: in notebook non serve il tracking, gli artefatti vanno in outputs/.
best_val_rmse = train(
    train_data=str(DATA_FILE),
    model_output=str(OUTPUT_DIR),
    window_size=30,
    batch_size=128,
    epochs=20,
    learning_rate=1e-3,
    val_ratio=0.2,
    max_rul=130,
    seed=42,
    use_mlflow=False,
)

print('Best val RMSE:', best_val_rmse)
print('Artefatti in:', OUTPUT_DIR)


## Valutazione sul test set

Esegue `src/evaluate.py` usando gli artefatti salvati in `outputs/` (`model.pt`, `scaler.pkl`).
Calcola MAE/RMSE sui RUL veri (`RUL_FD004.txt`) e salva `predictions.csv` ed `evaluation.json` in `outputs/`.


In [ ]:
import json
import subprocess

import pandas as pd

TEST_FILE = REPO_ROOT / 'CMAPPS-data' / 'test_FD004.txt'
RUL_LABELS = REPO_ROOT / 'CMAPPS-data' / 'RUL_FD004.txt'

cmd = [
    sys.executable, str(SRC_DIR / 'evaluate.py'),
    '--test_data', str(TEST_FILE),
    '--model_dir', str(OUTPUT_DIR),
    '--output_dir', str(OUTPUT_DIR),
    '--rul_labels', str(RUL_LABELS),
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(SRC_DIR))
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('evaluate.py terminato con errore')

with (OUTPUT_DIR / 'evaluation.json').open() as f:
    print('Metriche test:', json.load(f))

pd.read_csv(OUTPUT_DIR / 'predictions.csv').head(10)
